# clustering experiments

KMeans, DBSCAN, hierarchical. on iris and on a synthetic moons dataset.

In [1]:
from sklearn.datasets import load_iris, make_moons
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
%matplotlib inline

iris = load_iris()
Xs = StandardScaler().fit_transform(iris.data)

In [2]:
km = KMeans(n_clusters=3, random_state=0).fit(Xs)
plt.scatter(Xs[:,0], Xs[:,1], c=km.labels_)
plt.title('kmeans on iris (first 2 std features)')

## elbow plot

In [3]:
inertias = []
for k in range(1, 8):
    inertias.append(KMeans(n_clusters=k, random_state=0).fit(Xs).inertia_)
plt.plot(range(1,8), inertias, 'o-')
plt.xlabel('k'); plt.ylabel('inertia')

## DBSCAN on moons

In [4]:
Xm, _ = make_moons(n_samples=300, noise=0.1, random_state=0)
db = DBSCAN(eps=0.3, min_samples=5).fit(Xm)
plt.scatter(Xm[:,0], Xm[:,1], c=db.labels_)
plt.title('dbscan')

## agglomerative

In [5]:
ag = AgglomerativeClustering(n_clusters=3).fit(Xs)
plt.scatter(Xs[:,0], Xs[:,1], c=ag.labels_)
plt.title('hierarchical, k=3')

## dendrogram

In [6]:
from scipy.cluster.hierarchy import dendrogram, linkage
Z = linkage(Xs, method='ward')
plt.figure(figsize=(10,4))
dendrogram(Z, truncate_mode='level', p=4)
plt.title('dendro')

## cluster id as feature for classifier

In [7]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
feat = np.c_[Xs, KMeans(n_clusters=3, random_state=0).fit_predict(Xs)]
lr = LogisticRegression(solver='lbfgs', multi_class='auto', max_iter=300)
print('with cluster:', cross_val_score(lr, feat, iris.target, cv=5).mean())
print('without:', cross_val_score(lr, Xs, iris.target, cv=5).mean())

## takeaway
KMeans is fastest, DBSCAN handles non-convex but eps is fiddly.

## kmeans on digits, look at cluster centers as images

In [8]:
from sklearn.datasets import load_digits
d = load_digits()
km = KMeans(n_clusters=10, random_state=0).fit(d.data)
fig, axes = plt.subplots(2, 5, figsize=(8,4))
for c, ax in zip(km.cluster_centers_, axes.flat):
    ax.imshow(c.reshape(8,8), cmap='gray_r')
    ax.axis('off')